In [2]:
import pandas as pd
import numpy as np
import datetime as dt

df = pd.read_csv('online_retail.csv', encoding='ISO-8859-1')

df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['TotalSum'] = df['Quantity'] * df['UnitPrice']
df['CustomerID'] = df['CustomerID'].astype(int)

print("Data Shape:", df.shape)
print(df.head())

Data Shape: (397884, 9)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  TotalSum  
0  2010-12-01 08:26:00       2.55       17850  United Kingdom     15.30  
1  2010-12-01 08:26:00       3.39       17850  United Kingdom     20.34  
2  2010-12-01 08:26:00       2.75       17850  United Kingdom     22.00  
3  2010-12-01 08:26:00       3.39       17850  United Kingdom     20.34  
4  2010-12-01 08:26:00       3.39       17850  United Kingdom     20.34  


In [3]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby(['CustomerID']).agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalSum': 'sum'
})

rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSum': 'Monetary'
}, inplace=True)

print(rfm.head())

            Recency  Frequency  Monetary
CustomerID                              
12346           326          1  77183.60
12347             2        182   4310.00
12348            75         31   1797.24
12349            19         73   1757.55
12350           310         17    334.40


In [4]:
r_labels = range(5, 0, -1)
f_labels = range(1, 6)
m_labels = range(1, 6)

rfm['R'] = pd.qcut(rfm['Recency'], q=5, labels=r_labels)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=f_labels)
rfm['M'] = pd.qcut(rfm['Monetary'], q=5, labels=m_labels)

rfm['RFM_Group'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Score'] = rfm[['R', 'F', 'M']].sum(axis=1)

print(rfm.head())

            Recency  Frequency  Monetary  R  F  M RFM_Group  RFM_Score
CustomerID                                                            
12346           326          1  77183.60  1  1  5       115          7
12347             2        182   4310.00  5  5  5       555         15
12348            75         31   1797.24  2  3  4       234          9
12349            19         73   1757.55  4  4  4       444         12
12350           310         17    334.40  1  2  2       122          5


In [5]:
def segment_me(df):
    if df['RFM_Score'] >= 12:
        return 'VIP / Champions'
    elif (df['RFM_Score'] >= 9) and (df['RFM_Score'] < 12):
        return 'Loyal Customers'
    elif (df['RFM_Score'] >= 6) and (df['RFM_Score'] < 9):
        return 'Potential Loyalist'
    elif (df['RFM_Score'] >= 4) and (df['RFM_Score'] < 6):
        return 'At Risk / About to Sleep'
    else:
        return 'Hibernating / Lost'

rfm['Customer_Segment'] = rfm.apply(segment_me, axis=1)

print(rfm['Customer_Segment'].value_counts())
print("\n")
print(rfm.head())

Customer_Segment
VIP / Champions             1267
Potential Loyalist          1148
Loyal Customers             1041
At Risk / About to Sleep     624
Hibernating / Lost           258
Name: count, dtype: int64


            Recency  Frequency  Monetary  R  F  M RFM_Group  RFM_Score  \
CustomerID                                                               
12346           326          1  77183.60  1  1  5       115          7   
12347             2        182   4310.00  5  5  5       555         15   
12348            75         31   1797.24  2  3  4       234          9   
12349            19         73   1757.55  4  4  4       444         12   
12350           310         17    334.40  1  2  2       122          5   

                    Customer_Segment  
CustomerID                            
12346             Potential Loyalist  
12347                VIP / Champions  
12348                Loyal Customers  
12349                VIP / Champions  
12350       At Risk / About to Sleep 

In [6]:
rfm.to_csv('ECommerce_RFM_Analysis_Results.csv')